# 🎯 [MOV01_M02] 수송 수량 폭증에 따른 구간별 병목 지체 시간(Delay) 계산 전처리 모듈 구현 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV01/mov01_m02_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 3) |
| **미션 ID** | MOV01_M02 |
| **미션 주제** | 수송 수량 폭증에 따른 구간별 병목 지체 시간(Delay) 계산 전처리 모듈 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
수송 소요를 데이터로 예측해 군수 수송계획을 세우는 일은 구간별 통과 용량과 병목을 함께 보아야 정확해진다. 본 프로젝트는 수송 소요예측에서 예측 대상 정의, 시계열·피처 구성, 기준모델 비교, 설명가능성, 업무 임계값, 실적 대조를 익히도록 교육용 가상 노선망 자료를 코드로 만들어 구간별 수송 소요와 지체를 예측한다. 학습자는 병목 데이터 정의, 지체 전처리, BPR 기준 모델, 최대유량 기반 병목 식별과 피처 모델, 우회 소요·유류 산출과 앙상블 모델, 혼잡도 대시보드와 요인 설명, 작전군수관 승인(HITL)과 선형계획 최적화, 3개 구간 동시 차단 스트레스 시험, AAR 재보정과 ROI, 재학습 파이프라인 통합을 차례로 수행한다. 최종 목표는 구간별 소요·지체를 예측해 병목을 사전에 찾고 우회 물량 배분안을 만들되, 우회 명령의 하달 여부는 반드시 사람이 결정하는 수송 소요예측 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 야전(작전군수처, 수송대대)에 즉시 도입 가능한 "완성형 국방 AI 수송 시뮬레이터 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 기술 명세 및 운영 매뉴얼** | MSR 통과 능력 데이터 명세서(`docs/mov01_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mov01_operations_guide.md`) | 미션 1, 10 |
| **`src/`** | **군수 데이터 정제 및 분석 모듈** | 통과 지체시간 계산 전처리 파이프라인(`src/data_preprocessing.py`), BPR 기반 1차 지체 예측 Baseline(`src/baseline_delay.py`) | 미션 2, 3 |
| **`notebooks/`** | **통계 검증 및 심층 EDA** | MSR 구간 지체 탐색적 데이터 분석 및 파레토 분석(`notebooks/mov01_eda_analysis.ipynb`) | 미션 3 |
| **`src/`** | **AI 예측 & 우회로 탐색 엔진** | Max-Flow 기반 병목 사전 식별 및 우회로 탐색(`src/network_flow.py`), 유류 소모 및 소요시간 산출(`src/route_cost_engine.py`) | 미션 4, 5 |
| **`src/`** | **작전 상황판 & 스트레스 모의** | Streamlit 기반 전술 COP 대시보드(`src/dashboard_app.py`), 3개 축선 봉쇄 스트레스 시험(`src/stress_test.py`) | 미션 6, 8 |
| **`pipeline/`** | **HITL 통제 & 지속적 학습** | 작전군수관 3단계 승인 파이프라인(`pipeline/hitl_workflow.py`), 챔피언/챌린저 모델 자동 재학습 | 미션 7, 10 |
| **`reports/`** | **검증 및 AAR 리포트** | 3개 축선 봉쇄 스트레스 검증 보고서(`reports/mov01_stress_test_report.md`), AAR 오차 분석 및 ROI 보고서(`reports/mov01_roi_analysis.md`) | 미션 8, 9 |
| **`root`** | **통합 배포 패키지** | 통합 실행 진입점(`main.py`), 국방 망분리 Dockerfile 패키지 | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 수송 수량 폭증에 따른 구간별 병목 지체 시간(Delay) 계산 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

수송량 폭증에 따른 구간별 지체 시간 계산 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **수송량 폭증에 따른 구간별 지체 시간 계산을 위해 오염된 구간 통과 기록을 정제하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **결측, 음수, 단위 오류, 표기 오류, 미등록 구간, 용량 0, 중복을 주입한 가상 기록에 정제 규칙 10종을 순서대로 적용 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/etl/mov01_data_cleansing.py 및 정제 품질 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 특수 예외(비계획 정비, 긴급 불출 등) 데이터 보존의 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 규칙 10종 적용, 데이터 정합성 검증 통과율 100%


- 진행 기준: ETL 파이프라인 구현 및 정제된 마스터 데이터셋 구축


- 검증 방법: 전후 데이터 통계량(기술통계) 비교 및 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""수송량 폭증에 따른 구간별 지체 시간 계산 전처리 모듈: 오염된 구간 통과 기록에 정제 규칙 10종을 적용하고 비계획 정비·긴급 불출 같은 군수 특수 예외는 삭제하지 않고 보존하며, 정합성 검증과 전후 통계를 비교한다."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 노선망·구간 통과량 생성기 (실제 도로·지명·부대 없음)
# 가상 출발지(SRC)에서 가상 목적지(DST)까지 3개 축선(A·B·C)과 연결로 2개, 구간 10개. 150일치 피크시간 통과량·지체 시간을 만든다.
# ---------------------------------------------------------------------------
SEGS = pd.DataFrame([
    ("A1", "SRC", "J1", 12, 130, "일반"), ("A2", "J1", "J2", 8, 70, "교량"), ("A3", "J2", "DST", 10, 140, "교차로"),
    ("B1", "SRC", "J3", 15, 100, "일반"), ("B2", "J3", "J2", 9, 80, "터널"), ("B3", "J3", "DST", 14, 95, "일반"),
    ("C1", "SRC", "J4", 18, 70, "일반"), ("C2", "J4", "DST", 20, 75, "교량"),
    ("X1", "J1", "J3", 7, 50, "교차로"), ("X2", "J4", "J3", 10, 60, "일반")],
    columns=["seg", "u", "v", "km", "cap", "kind"])               # cap = 피크시간 통과 용량(대/시, 교육용 가정값)
KSPD = {"일반": 55, "교량": 40, "터널": 45, "교차로": 45}           # 구간 유형별 자유 주행 속도(km/h, 가정값)
SEGS["t0"] = SEGS["km"] / SEGS["kind"].map(KSPD) * 60              # 자유 주행 시간(분)
UTIL = dict(A1=.78, A2=.85, A3=.66, B1=.70, B2=.72, B3=.64, C1=.55, C2=.58, X1=.45, X2=.40)   # 평시 이용률


def bpr(t0, v, cap, a=0.15, b=4.0):
    """BPR 지체 함수(교통공학의 표준형): 통과량/용량 비가 커질수록 지체가 급증한다. (분)"""
    # [TODO : 학생 실습 2] 수송 수량 폭증에 따른 구간별 병목 지체 시간(Delay) 계산 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_data(seed=11, days=150):
    """구간 x 일자 패널. v=실제 피크 통과량, plan_v=수송계획 수량(긴급 불출 미반영), delay=실제 지체(분)."""
    rng = np.random.default_rng(seed)
    d_idx = np.arange(days)
    lvl = np.array([1.0, 1.08, 1.04, 1.10, 1.02, 0.82, 0.78])[d_idx % 7] * (1 + 0.0012 * d_idx)
    lvl = lvl * np.where((d_idx >= 100) & (d_idx < 110), 1.30, 1.0)       # 수송량 폭증 기간(가상)
    prev, rows = {s: 0.0 for s in SEGS["seg"]}, []
    for d in d_idx:
        rain = bool(rng.random() < 0.18)
        for r in SEGS.itertuples():
            maint, urgent = bool(rng.random() < 0.06), bool(rng.random() < 0.05)   # 비계획 정비 / 긴급 불출
            v = r.cap * UTIL[r.seg] * lvl[d] * (1.25 if urgent else 1.0) * rng.normal(1, 0.05)
            cap_eff = r.cap * (0.65 if maint else 1.0) * (0.92 if rain else 1.0)
            delay = max(r.t0 * 0.2 * (v / cap_eff) ** 3.5 + 0.25 * prev[r.seg] + rng.normal(0, 0.4), 0.0)
            prev[r.seg] = delay
            rows.append(dict(day=int(d), dow=int(d % 7), seg=r.seg, kind=r.kind, km=r.km, cap=r.cap, t0=r.t0,
                             plan_v=v / (1.25 if urgent else 1.0) * rng.normal(1, 0.04), v=v, maint=int(maint), urgent=int(urgent),
                             rain=int(rain), cap_eff=cap_eff, delay=delay))
    df = pd.DataFrame(rows)
    df["transit"] = df["t0"] * (1 + 0.08 * df["rain"]) + df["delay"]
    return df


def mape(y, p):
    # [TODO : 학생 실습 2] 수송 수량 폭증에 따른 구간별 병목 지체 시간(Delay) 계산 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))
# ---------------------------------------------------------------------------

clean = make_data()
rng = np.random.default_rng(2)
dirty = clean.copy()
# ---- 오염 주입(가상): 결측·음수·단위 오류·표기 오류·미등록 구간·0 용량·중복 ----
ix = rng.choice(len(dirty), 40, replace=False)
dirty.loc[ix[:12], "v"] = np.nan
dirty.loc[ix[12:18], "v"] = -dirty.loc[ix[12:18], "v"]
dirty.loc[ix[18:23], "v"] = dirty.loc[ix[18:23], "v"] * 10
dirty.loc[ix[23:29], "transit"] = np.nan
dirty.loc[ix[29:32], "cap"] = 0
dirty.loc[ix[32:36], "seg"] = dirty.loc[ix[32:36], "seg"].str.lower() + " "
ghost = dirty.iloc[ix[36:40]].copy(); ghost["seg"] = "Z9"                      # 마스터에 없는 구간 코드가 섞여 들어온 행
dirty = pd.concat([dirty, ghost, dirty.iloc[rng.choice(len(dirty), 15, replace=False)]], ignore_index=True)
planted = {"결측 통과량": 12, "음수 통과량": 6, "단위 오류(10배)": 5, "결측 통과시간": 6, "용량 0": 3, "구간 표기 오류": 4, "미등록 구간": 4, "중복 행": 15}
print(f"[0] 오염 데이터 {len(dirty)}행 (원본 {len(clean)}행), 주입 오류 {sum(planted.values())}건: {planted}")

log, w = [], dirty.copy()
master = SEGS.set_index("seg")


def rule(name, n):
    log.append((len(log) + 1, name, n))


# R1 구간 코드 표기 정규화
before = w["seg"].copy(); w["seg"] = w["seg"].str.strip().str.upper(); rule("구간 코드 공백·대소문자 정규화", int((before != w["seg"]).sum()))
# R2 미등록 구간 격리(삭제하지 않고 별도 보관)
bad = ~w["seg"].isin(master.index); quarantine = w[bad].copy(); w = w[~bad]; rule("미등록 구간 격리(별도 보관)", int(bad.sum()))
# R3 중복 제거(구간·일자 키)
n0 = len(w); w = w.drop_duplicates(["seg", "day"], keep="first"); rule("(구간, 일자) 중복 제거", n0 - len(w))
# R4 용량 0/결측은 구간 마스터에서 복원
bad = w["cap"] <= 0; w.loc[bad, "cap"] = w.loc[bad, "seg"].map(master["cap"]); rule("용량 0 -> 구간 마스터 복원", int(bad.sum()))
# R5 음수 통과량은 결측으로 전환(부호 오류 가능성 -> 보간 대상)
bad = w["v"] < 0; w.loc[bad, "v"] = np.nan; rule("음수 통과량 -> 결측 처리", int(bad.sum()))
# R6 단위 오류(용량의 3배 초과)는 1/10 보정
bad = w["v"] > 3 * w["cap"]; w.loc[bad, "v"] = w.loc[bad, "v"] / 10; rule("단위 오류(용량 3배 초과) 1/10 보정", int(bad.sum()))
# R7 결측 통과량: 같은 구간·같은 요일의 중앙값으로 보간(계절성 보간)
bad = w["v"].isna(); med = w.groupby(["seg", "dow"])["v"].transform("median"); w.loc[bad, "v"] = med[bad]; rule("결측 통과량 -> 구간·요일 중앙값 보간", int(bad.sum()))
# R8 결측 통과시간: 자유주행 + 지체로 재계산
bad = w["transit"].isna(); w.loc[bad, "transit"] = w.loc[bad, "t0"] * (1 + 0.08 * w.loc[bad, "rain"]) + w.loc[bad, "delay"]; rule("결측 통과시간 재계산", int(bad.sum()))
# R9 지체 시간 정합: delay = transit - 자유주행(우천 보정), 음수는 0으로
d2 = (w["transit"] - w["t0"] * (1 + 0.08 * w["rain"])).clip(lower=0); mism = (d2 - w["delay"]).abs() > 1e-6
w["delay"] = d2; rule("지체 = 통과시간 - 자유주행 정합 재계산", int(mism.sum()))
# R10 이상치 표시: IQR 밖 지체는 '표시'만 하고, 정비·긴급 불출 행은 예외로 보존(삭제 금지)
q1, q3 = w["delay"].quantile([.25, .75]); out = w["delay"] > q3 + 3 * (q3 - q1)
w["outlier"] = out.astype(int); w["exception_keep"] = ((w["maint"] == 1) | (w["urgent"] == 1)).astype(int)
rule("지체 이상치 플래그(삭제 없음)", int(out.sum()))
w = w.sort_values(["day", "seg"]).reset_index(drop=True)

print("\n[1] 정제 규칙 적용 로그")
print(pd.DataFrame(log, columns=["규칙", "내용", "처리 건수"]).to_string(index=False))
print(f"    정제 규칙 {len(log)}종 적용(목표 10종: {'달성' if len(log) >= 10 else '미달'})")

# ---- 정합성 검증 ----
checks = {"(구간, 일자) 유일": not w.duplicated(["seg", "day"]).any(), "필수 컬럼 결측 0": int(w[["seg", "v", "cap", "delay", "transit"]].isna().sum().sum()) == 0,
          "구간 코드 마스터 포함": bool(w["seg"].isin(master.index).all()), "통과량 0 이상·용량 3배 이하": bool(((w["v"] >= 0) & (w["v"] <= 3 * w["cap"])).all()),
          "용량 = 마스터 값": bool((w["cap"] == w["seg"].map(master["cap"])).all()), "통과시간 >= 자유주행": bool((w["transit"] >= w["t0"] - 1e-9).all()),
          "지체 >= 0": bool((w["delay"] >= 0).all())}
passed = sum(checks.values())
print("\n[2] 정합성 검증")
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(f"    통과율 {passed / len(checks):.0%} (목표 100%)")

# ---- 군수 특수 예외 보존 / 원본 대비 복원 정확도 ----
ex = w[w["exception_keep"] == 1]
print(f"\n[3] 군수 특수 예외 보존: 비계획 정비 {int(w['maint'].sum())}건, 긴급 불출 {int(w['urgent'].sum())}건 모두 유지(원본 {int(clean['maint'].sum())}, {int(clean['urgent'].sum())}건)")
print(f"    예외 행 평균 지체 {ex['delay'].mean():.2f}분 vs 일반 행 {w.loc[w['exception_keep'] == 0, 'delay'].mean():.2f}분 -> 이상치로 지웠다면 폭증 구간 정보가 사라진다")
m = w.merge(clean[["seg", "day", "v"]], on=["seg", "day"], suffixes=("", "_true"))
print(f"    통과량 복원 오차(보정·보간된 행 {int((m['v'] != m['v_true']).sum())}건 포함 전체 MAE) {np.abs(m['v'] - m['v_true']).mean():.3f}대/시")
print("\n[4] 전후 기술통계 비교(통과량 v / 지체 delay)")
st = pd.DataFrame({"오염 원본": dirty[["v", "delay"]].describe().loc[["count", "mean", "min", "max"]].stack(), "정제 후": w[["v", "delay"]].describe().loc[["count", "mean", "min", "max"]].stack(),
                   "정답 원본": clean[["v", "delay"]].describe().loc[["count", "mean", "min", "max"]].stack()}).round(2)
print(st.to_string())


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[0] 오염 데이터 1519행 (원본 1500행), 주입 오류 55건: {'결측 통과량': 12, '음수 통과량': 6, '단위 오류(10배)': 5, '결측 통과시간': 6, '용량 0': 3, '구간 표기 오류': 4, '미등록 구간': 4, '중복 행': 15}

[1] 정제 규칙 적용 로그
 규칙                      내용  처리 건수
  1       구간 코드 공백·대소문자 정규화      4
  2        미등록 구간 격리(별도 보관)      4
  3          (구간, 일자) 중복 제거     15
  4       용량 0 -> 구간 마스터 복원      3
  5         음수 통과량 -> 결측 처리      6
  6 단위 오류(용량 3배 초과) 1/10 보정      5
  7  결측 통과량 -> 구간·요일 중앙값 보간     18
  8             결측 통과시간 재계산      6
  9 지체 = 통과시간 - 자유주행 정합 재계산      0
 10       지체 이상치 플래그(삭제 없음)     43
    정제 규칙 10종 적용(목표 10종: 달성)

[2] 정합성 검증
  [PASS] (구간, 일자) 유일
  [PASS] 필수 컬럼 결측 0
  [PASS] 구간 코드 마스터 포함
  [PASS] 통과량 0 이상·용량 3배 이하
  [PASS] 용량 = 마스터 값
  [PASS] 통과시간 >= 자유주행
  [PASS] 지체 >= 0
    통과율 100% (목표 100%)

[3] 군수 특수 예외 보존: 비계획 정비 95건, 긴급 불출 87건 모두 유지(원본 95, 87건)
    예외 행 평균 지체 4.37분 vs 일반 행 1.60분 -> 이상치로 지웠다면 폭증 구간 정보가 사라진다
    통과량 복원 오차(보정·보간된 행 18건 포함 전체 MAE) 0.104대/시

[4] 전후 기술통계 비교(통과량 v / 지체 delay)
               오염 원본     정제 후    정답 원본
count v      1507.00  1500.00  1500.00
      delay  1519.00  1500.00  1500.00
mean  v        63.64    62.95    62.95
      delay     1.94     1.93     1.93
min   v      -146.18    16.78    16.78
      delay     0.00     0.00     0.00
max   v       635.70   174.46   174.46
      delay    21.79    21.79    21.79
```
</details>
